In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def initial_1d(N):
    return 2 * np.random.randint(0, 2, N) - 1

def mcmove_1d(config, beta):
    N = len(config)
    i = np.random.randint(N)
    s = config[i]
    nb = config[(i+1)%N] + config[(i-1)%N]
    dE = 2 * s * nb
    if dE < 0 or np.random.rand() < np.exp(-dE * beta):
        config[i] = -s

def energy_1d(config):
    return -np.sum(config * np.roll(config, 1))

def mag_1d(config):
    return np.sum(config)

def initial_2d(N):
    return 2 * np.random.randint(0, 2, (N, N)) - 1

def mcmove_2d(config, beta):
    N = len(config)
    i, j = np.random.randint(0, N, 2)
    s = config[i, j]
    nb = config[(i+1)%N, j] + config[(i-1)%N, j] + config[i, (j+1)%N] + config[i, (j-1)%N]
    dE = 2 * s * nb
    if dE < 0 or np.random.rand() < np.exp(-dE * beta):
        config[i, j] = -s

def energy_2d(config):
    return -np.sum(config * np.roll(config, 1, axis=0) + config * np.roll(config, 1, axis=1))

def mag_2d(config):
    return np.sum(config)

def calculate_equil_time(energy_list, threshold=0.05):
    if len(energy_list) < 40:
        return len(energy_list) * 10
    half_idx = len(energy_list) // 2
    for i in range(half_idx, len(energy_list)):
        window = energy_list[i-20:i]
        var = np.var(window)
        avg = np.mean(window)
        if avg != 0 and var < threshold:
            return i * 10
    return len(energy_list) * 10

def equil_1d(N, steps, T):
    cfg = initial_1d(N)
    b = 1 / T
    e_list = []
    for s in range(steps):
        mcmove_1d(cfg, b)
        if s % 10 == 0:
            e_list.append(energy_1d(cfg))
    return e_list

def equil_2d(N, steps, T):
    cfg = initial_2d(N)
    b = 1 / T
    e_list = []
    for s in range(steps):
        mcmove_2d(cfg, b)
        if s % 10 == 0:
            e_list.append(energy_2d(cfg))
    return e_list

def run_1d(N, T_range, eq, mc):
    M, E, C, X = [], [], [], []
    for T in T_range:
        cfg = initial_1d(N)
        b = 1 / T
        for _ in range(eq):
            mcmove_1d(cfg, b)
        E1, E2, M1, M2 = 0, 0, 0, 0
        for _ in range(mc):
            mcmove_1d(cfg, b)
            en, mg = energy_1d(cfg), mag_1d(cfg)
            E1 += en
            E2 += en ** 2
            M1 += mg
            M2 += mg ** 2
        n1 = 1 / (mc * N)
        n2 = 1 / (mc ** 2 * N)
        M.append(abs(n1 * M1))
        E.append(n1 * E1)
        C.append((n1 * E2 - n2 * E1 ** 2) / (T ** 2))
        X.append((n1 * M2 - n2 * M1 ** 2) / T)
    return M, E, C, X

def run_2d(N, T_range, eq, mc):
    M, E, C, X = [], [], [], []
    for T in T_range:
        cfg = initial_2d(N)
        b = 1 / T
        for _ in range(eq):
            mcmove_2d(cfg, b)
        E1, E2, M1, M2 = 0, 0, 0, 0
        for _ in range(mc):
            mcmove_2d(cfg, b)
            en, mg = energy_2d(cfg), mag_2d(cfg)
            E1 += en
            E2 += en ** 2
            M1 += mg
            M2 += mg ** 2
        n1 = 1 / (mc * N * N)
        n2 = 1 / (mc ** 2 * N * N)
        M.append(abs(n1 * M1))
        E.append(n1 * E1)
        C.append((n1 * E2 - n2 * E1 ** 2) / (T ** 2))
        X.append((n1 * M2 - n2 * M1 ** 2) / T)
    return M, E, C, X

N = 16
T1 = np.linspace(0.5, 2.0, 12)
T2 = np.linspace(2.0, 2.5, 24)
T3 = np.linspace(2.5, 3.5, 8)
T_range = np.concatenate([T1, T2[1:], T3[1:]])
Tc = 2.269
eq1, mc1 = 2000, 3000
eq2, mc2 = 28000, 35000  
eq_1d = equil_1d(N, eq1, 2.5)
eq_2d = equil_2d(N, eq2, 2.5)
M1, E1, C1, X1 = run_1d(N, T_range, eq1, mc1)
M2, E2, C2, X2 = run_2d(N, T_range, eq2, mc2)

print("="*80)
print("                     1D & 2D Ising Model Key Simulation Data Output")
print("="*80)

print("\n[1. Simulation Parameters Details]")
print(f"Lattice Size N: {N}")
print(f"Temperature Range: {np.round(T_range[0], 2)} ~ {np.round(T_range[-1], 2)} (Total {len(T_range)} temperature points)")
print(f"2D Critical Temperature Tc: {Tc}")
print(f"1D Equilibration Steps eq1: {eq1}, Simulation Steps mc1: {mc1}")
print(f"2D Equilibration Steps eq2: {eq2}, Simulation Steps mc2: {mc2}")

eq_time_1d = calculate_equil_time(eq_1d)
eq_time_2d = calculate_equil_time(eq_2d)
print("\n[2. Equilibration Time (MCS)]")
print(f"1D Ising Model Equilibration Time: {eq_time_1d} MCS")
print(f"2D Ising Model Equilibration Time: {eq_time_2d} MCS")

print("\n[3. 1D Ising Model Thermodynamic Parameters (Sorted by Temperature)]")
print(f"{'Temperature T':<12} {'|Magnetization|':<18} {'Average Energy':<18} {'Specific Heat Cv':<18} {'Susceptibility':<18}")
print("-"*80)
for i in range(len(T_range)):
    print(f"{np.round(T_range[i], 2):<12} {np.round(M1[i], 4):<18} {np.round(E1[i], 4):<18} {np.round(C1[i], 4):<18} {np.round(X1[i], 4):<18}")

print("\n[4. 2D Ising Model Thermodynamic Parameters (Sorted by Temperature, Tc≈2.27)]")
print(f"{'Temperature':<12} {'|Magnetization|':<18} {'Average Energy E':<18} {'Specific Heat':<18} {'Susceptibility χ':<18}")
print("-"*80)
for i in range(len(T_range)):
    t = np.round(T_range[i], 2)
    m = np.round(M2[i], 4)
    e = np.round(E2[i], 4)
    c = np.round(C2[i], 4)
    x = np.round(X2[i], 4)
    if abs(t - Tc) < 0.1:
        print(f"{t:<12} {m:<18} {e:<18} {c:<18} {x:<18}  <-- Near Critical Temperature")
    else:
        print(f"{t:<12} {m:<18} {e:<18} {c:<18} {x:<18}")

print("\n[5. Key Extreme Values]")
print(f"1D Ising Model - Maximum Specific Heat: {np.round(max(C1), 4)} (Temperature: {np.round(T_range[np.argmax(C1)], 2)})")
print(f"1D Ising Model - Maximum Susceptibility: {np.round(max(X1), 4)} (Temperature: {np.round(T_range[np.argmax(X1)], 2)})")
print(f"2D Ising Model - Maximum Specific Heat: {np.round(max(C2), 4)} (Temperature: {np.round(T_range[np.argmax(C2)], 2)})")
print(f"2D Ising Model - Maximum Susceptibility: {np.round(max(X2), 4)} (Temperature: {np.round(T_range[np.argmax(X2)], 2)})")
print(f"2D Ising Model - Maximum Magnetization: {np.round(max(M2), 4)} (Temperature: {np.round(T_range[np.argmax(M2)], 2)})")
print("="*80)

plt.figure(figsize=(10, 4))
plt.subplot(121); plt.plot(eq_1d, 'b-', linewidth=1.5); plt.title("1D Equilibration")
plt.xlabel("Step / 10"); plt.ylabel("Energy")
plt.subplot(122); plt.plot(eq_2d, 'r-', linewidth=1.5); plt.title("2D Equilibration")
plt.xlabel("Step / 10"); plt.ylabel("Energy")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, 2, figsize=(10, 7))
ax[0, 0].plot(T_range, M1, 'ro-'); ax[0, 0].set_title("1D |Magnetization|")
ax[0, 1].plot(T_range, E1, 'bo-'); ax[0, 1].set_title("1D Energy")
ax[1, 0].plot(T_range, C1, 'go-'); ax[1, 0].set_title("1D Specific Heat")
ax[1, 1].plot(T_range, X1, 'mo-'); ax[1, 1].set_title("1D Susceptibility")
for a in ax.flat:
    a.set_xlabel("T")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, 2, figsize=(10, 7))
ax[0, 0].plot(T_range, M2, 'ro-'); ax[0, 0].axvline(x=Tc, c='k', ls='--')
ax[0, 0].set_title("2D |Magnetization|")
ax[0, 1].plot(T_range, E2, 'bo-'); ax[0, 1].axvline(x=Tc, c='k', ls='--')
ax[0, 1].set_title("2D Energy")
ax[1, 0].plot(T_range, C2, 'go-'); ax[0, 0].axvline(x=Tc, c='k', ls='--')
ax[1, 0].set_title("2D Specific Heat")
ax[1, 1].plot(T_range, X2, 'mo-'); ax[1, 1].axvline(x=Tc, c='k', ls='--')
ax[1, 1].set_title("2D Susceptibility")
for a in ax.flat:
    a.set_xlabel("T")
plt.tight_layout(); plt.show()